# EDA и ML-отчет по Lakehouse

Notebook рассчитан на запуск после основного пайплайна: Bronze, Silver, Gold и ML feature table уже должны существовать в папке `data/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import polars as pl
from deltalake import DeltaTable

from src.config import BRONZE_PATH, SILVER_PATH, GOLD_PATH

def delta_summary(name, path):
    if not DeltaTable.is_deltatable(str(path)):
        return {'table': name, 'exists': False, 'rows': 0, 'version': None}
    rows = pl.scan_delta(str(path)).select(pl.len()).collect().item()
    return {'table': name, 'exists': True, 'rows': rows, 'version': DeltaTable(str(path)).version()}

tables = [
    delta_summary('bronze', BRONZE_PATH),
    delta_summary('silver', SILVER_PATH),
    delta_summary('gold_ml_features', GOLD_PATH / 'ml_features'),
]
pl.DataFrame(tables)

## Обзор Silver-слоя

In [ ]:
silver = pl.scan_delta(str(SILVER_PATH))

overview = silver.select([
    pl.len().alias('rows'),
    pl.col('flight_date').min().alias('min_flight_date'),
    pl.col('flight_date').max().alias('max_flight_date'),
    pl.col('airline').n_unique().alias('airlines'),
    pl.col('origin').n_unique().alias('origins'),
    pl.col('dest').n_unique().alias('destinations'),
    pl.col('route').n_unique().alias('routes'),
    pl.col('arr_delay').mean().alias('mean_arr_delay'),
    pl.col('arr_delay').median().alias('median_arr_delay'),
    pl.col('arr_delay').quantile(0.95).alias('p95_arr_delay'),
    (pl.col('arr_delay') > 15).mean().alias('delay_rate'),
]).collect()

overview

## Распределение задержек

In [ ]:
delay_histogram = (
    silver.select(pl.col('arr_delay'))
    .with_columns(
        pl.when(pl.col('arr_delay') < -15).then(pl.lit('< -15'))
        .when(pl.col('arr_delay') < 0).then(pl.lit('-15..0'))
        .when(pl.col('arr_delay') <= 15).then(pl.lit('0..15'))
        .when(pl.col('arr_delay') <= 60).then(pl.lit('16..60'))
        .when(pl.col('arr_delay') <= 180).then(pl.lit('61..180'))
        .otherwise(pl.lit('> 180'))
        .alias('delay_bucket')
    )
    .group_by('delay_bucket')
    .agg(pl.len().alias('flights'))
    .sort('delay_bucket')
    .collect()
)

delay_histogram

## Аналитические Gold-витрины

In [ ]:
airline_delay = pl.scan_delta(str(GOLD_PATH / 'analytics' / 'airline_delay')).collect()
airport_delay = pl.scan_delta(str(GOLD_PATH / 'analytics' / 'airport_delay')).collect()
route_delay = pl.scan_delta(str(GOLD_PATH / 'analytics' / 'route_delay')).collect()
time_delay = pl.scan_delta(str(GOLD_PATH / 'analytics' / 'time_delay')).collect()

airline_delay.sort('avg_arr_delay', descending=True).head(10)

In [ ]:
route_delay.sort('avg_arr_delay', descending=True).head(10)

In [ ]:
time_delay.sort(['day_of_week', 'hour']).head(24)

## План Lazy-запроса

План ниже показывает projection pushdown и selection pushdown для типичного аналитического запроса.

In [ ]:
query = (
    pl.scan_delta(str(SILVER_PATH))
    .filter(pl.col('arr_delay') > 15)
    .select(['airline', 'origin', 'hour', 'arr_delay'])
    .group_by(['airline', 'origin', 'hour'])
    .agg(pl.col('arr_delay').mean().alias('avg_arr_delay'))
)

print(query.explain())

## Benchmark моделей в notebook

Компактный benchmark добавлен прямо в notebook, чтобы отчет оставался воспроизводимым даже без открытия MLflow UI.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ml_df = pl.scan_delta(str(GOLD_PATH / 'ml_features')).collect()
if len(ml_df) > 50000:
    ml_df = ml_df.sample(n=50000, seed=42)

pdf = ml_df.to_pandas()
numeric = [c for c in ['distance', 'hour', 'day_of_week', 'origin_day_flights', 'dest_day_flights', 'airline_day_flights', 'route_day_flights', 'origin_hour_flights', 'route_hour_flights'] if c in pdf.columns]
categorical = [c for c in ['airline', 'origin', 'dest', 'season', 'route'] if c in pdf.columns]
features = numeric + categorical

preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])

X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    pdf[features], pdf['arr_delay'], pdf['is_delayed'], test_size=0.2, random_state=42,
    stratify=pdf['is_delayed'] if pdf['is_delayed'].nunique() > 1 else None,
)

regression_models = {
    'DummyMean': DummyRegressor(strategy='mean'),
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=40, max_depth=6, random_state=42, n_jobs=-1),
}

regression_results = []
for name, estimator in regression_models.items():
    model = Pipeline([('preprocess', preprocess), ('model', estimator)])
    model.fit(X_train, y_train_reg)
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test_reg, pred)
    regression_results.append({'model': name, 'MAE': mean_absolute_error(y_test_reg, pred), 'RMSE': mse ** 0.5, 'R2': r2_score(y_test_reg, pred)})

pd.DataFrame(regression_results).sort_values('RMSE')

In [ ]:
classification_models = {
    'DummyMostFrequent': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=40, max_depth=6, random_state=42, n_jobs=-1),
}

classification_results = []
for name, estimator in classification_models.items():
    model = Pipeline([('preprocess', preprocess), ('model', estimator)])
    model.fit(X_train, y_train_clf)
    pred = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else pred
    try:
        auc = roc_auc_score(y_test_clf, score)
    except ValueError:
        auc = 0.5
    classification_results.append({
        'model': name,
        'accuracy': accuracy_score(y_test_clf, pred),
        'precision': precision_score(y_test_clf, pred, zero_division=0),
        'recall': recall_score(y_test_clf, pred, zero_division=0),
        'F1': f1_score(y_test_clf, pred, zero_division=0),
        'ROC_AUC': auc,
    })

pd.DataFrame(classification_results).sort_values('F1', ascending=False)